In [5]:
from src.data_loader import DataLoader
from src.power_station import PowerStation
from src.load_forecaster import LoadForecaster
from src.outage_analyzer import OutageAnalyzer
from src.risk_model import RiskModel

loader = DataLoader("live_grid_load.csv", "data/raw/weather_hudson_station.csv", "data/raw/project_electrical_outages.csv")
merged = loader.merge_all()

station = PowerStation("Hudson Substation", 40.728, -74.078, rated_capacity=85)
station.load_history = merged['Load_Percent'].tolist()

forecaster = LoadForecaster(merged)
forecast_hour = forecaster.forecast_next_hour()

outages = loader.load_outage_data()
out_analyzer = OutageAnalyzer(outages)
out_prob = out_analyzer.compute_outage_probability(merged['Timestamp'].iloc[-1])

risk_engine = RiskModel(station.rated_capacity)
risk_score = risk_engine.compute_risk_score(forecast_hour, out_prob, merged['temperature_C'].iloc[-1])
risk_level = risk_engine.classify(risk_score)

print("Next-hour forecast:", forecast_hour)
print("Outage probability:", out_prob)
print("Risk score:", risk_score, "=>", risk_level)

ModuleNotFoundError: No module named 'src'

In [6]:
import matplotlib.pyplot as plt
import pandas as pd

# ---------------------------------------------------------------------------
def plot_load_over_time(merged_df):
    """
    Plot: Grid Load Over Time.

    Purpose:
        Shows how load (%) changes over time. Helps identify daily cycles and system peaks.
   """
    plt.figure(figsize=(12, 5))
    plt.plot(merged_df['Timestamp'], merged_df['Load_Percent'],
             label="Load (%)", color="blue")

    plt.title("Grid Load Over Time")
    plt.xlabel("Timestamp")
    plt.ylabel("Load Percent")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [7]:
def plot_outages_by_month(outages_df):
    """
    Bar Plot: Outage Frequency by Month.

    Purpose:
        Uses outage timestamps to compute and visualize how many outages
        occur in each month. Helps expose seasonal risks.
    """
    outages_df['month'] = outages_df['start_time'].dt.month
    monthly_counts = outages_df.groupby('month').size()

    plt.figure(figsize=(10, 5))
    monthly_counts.plot(kind='bar', color='red')

    plt.title("Outages by Month")
    plt.xlabel("Month")
    plt.ylabel("Number of Outages")
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()

In [8]:
def plot_risk_over_time(merged_df, outage_analyzer, risk_engine):
    """
    Plot: Risk Score Over Time.

    Purpose:
        Computes risk score over the full timeline using:
            - Load percent
            - Outage probability
            - Temperature
       
    outage_analyzer: OutageAnalyzer instance
    risk_engine: RiskModel instance
    """
    risk_scores = []

    for _, row in merged_df.iterrows():
        load = row['Load_Percent']
        temp = row['Temperature_C']
        ts = row['Timestamp']
        prob = outage_analyzer.compute_outage_probability(ts)
        risk_scores.append(risk_engine.compute_risk_score(load, prob, temp))

    plt.figure(figsize=(12, 5))
    plt.plot(merged_df['Timestamp'], risk_scores, color='green')

    plt.title("Risk Score Over Time")
    plt.xlabel("Timestamp")
    plt.ylabel("Risk Score")
    plt.grid(True)
    plt.tight_layout()
    plt.show()